# Dataset Evaluation: Clean vs Poisoned Data

This notebook evaluates simple sklearn models on:
- Clean datasets
- AR (At Random) poisoned datasets
- NAR (Not At Random) poisoned datasets

We test 3-4 efficient classifiers/regressors and measure performance degradation across multiple random seeds (42-46) using parallelized testing with joblib.

In [1]:
import pandas as pd
import numpy as np
import warnings
from joblib import Parallel, delayed
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score

# Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# Import data loading utilities
import sys
sys.path.insert(0, '..')
from frogdq.data import get_datasets, load_data

warnings.filterwarnings("ignore")

## Configuration

In [2]:
# Test seeds to use
SEEDS = list(range(42, 47))  # Seeds 42 to 46

# Data modes to test
MODES = ["clean", "ar", "nar"]

# Number of parallel jobs for joblib
N_JOBS = -1  # Use all available cores

print(f"Testing on seeds: {SEEDS}")
print(f"Testing modes: {MODES}")
print(f"Parallel jobs: {N_JOBS if N_JOBS > 0 else 'all cores'}")

Testing on seeds: [42, 43, 44, 45, 46]
Testing modes: ['clean', 'ar', 'nar']
Parallel jobs: all cores


## Load Available Datasets

In [4]:
# Get list of all available datasets
datasets_df = get_datasets(data_dir="../data", poisoned_dir="../data_poisoned")

# Filter to only datasets that have both AR and NAR poisoned versions
available_datasets = datasets_df[
    (datasets_df["has_ar"] == True) & (datasets_df["has_nar"] == True)
].copy()

print(f"Found {len(available_datasets)} datasets with all poisoning modes")
print(f"\nDatasets to evaluate:")
display(available_datasets[['dataset_name', 'task_type', 'n_samples', 'n_features', 'n_classes']])

Found 50 datasets with all poisoning modes

Datasets to evaluate:


,dataset_name,task_type,n_samples,n_features,n_classes
0,iris,classification,150,4,3
1,heart_disease,regression,303,13,0
2,wine_quality,regression,6497,11,0
3,breast_cancer_wisconsin_diagnostic,classification,569,30,2
4,bank_marketing,classification,45211,16,2
5,adult,classification,48842,14,4
6,wine,classification,178,13,3
7,car_evaluation,classification,1728,6,4
8,predict_students_dropout_and_academic_success,classification,4424,36,3
9,default_of_credit_card_clients,classification,30000,23,2


## Model Selection Functions

In [5]:
def get_models(task_type, seed=42):
    """Get simple, efficient models for the task with specific seed."""
    if task_type == "classification":
        return {
            "LogisticRegression": LogisticRegression(max_iter=1000, random_state=seed),
            "DecisionTree": DecisionTreeClassifier(max_depth=10, random_state=seed),
            "RandomForest": RandomForestClassifier(
                n_estimators=100, max_depth=10, random_state=seed, n_jobs=1
            ),
            "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=1),
        }
    else:  # regression
        return {
            "Ridge": Ridge(random_state=seed),
            "DecisionTree": DecisionTreeRegressor(max_depth=10, random_state=seed),
            "RandomForest": RandomForestRegressor(
                n_estimators=100, max_depth=10, random_state=seed, n_jobs=1
            ),
            "KNN": KNeighborsRegressor(n_neighbors=5, n_jobs=1),
        }


def evaluate_model(model, X_train, X_test, y_train, y_test, task_type):
    """Train and evaluate a single model."""
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        if task_type == "classification":
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
            return {"accuracy": acc, "f1_score": f1}
        else:  # regression
            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            return {"mse": mse, "r2_score": r2}
    except Exception as e:
        print(f"Error evaluating model: {e}")
        if task_type == "classification":
            return {"accuracy": 0.0, "f1_score": 0.0}
        else:
            return {"mse": float("inf"), "r2_score": -float("inf")}

## Evaluation Function with Parallelization

In [8]:
def evaluate_single_configuration(dataset_name, mode, seed, task_type):
    """Evaluate all models for a single dataset-mode-seed configuration."""
    results = []
    
    try:
        # Load data using the data.py utilities
        # Always use clean test set for fair comparison
        (X_train, X_val, X_test), (y_train, y_val, y_test), preprocessor, metadata = load_data(
            dataset_name=dataset_name,
            mode=mode,
            seed=seed,
            clean_val=True,  # Use clean validation
            clean_test=True,  # Use clean test for fair comparison
            data_dir="../data",
            poisoned_dir="../data_poisoned",
        )
        
        # Get models for this task type and seed
        models = get_models(task_type, seed=seed)
        
        # Evaluate each model
        for model_name, model in models.items():
            metrics = evaluate_model(model, X_train, X_test, y_train, y_test, task_type)
            
            results.append({
                "dataset": dataset_name,
                "task_type": task_type,
                "data_mode": mode,
                "seed": seed,
                "model": model_name,
                "overall_quality": metadata["overall_quality"],
                "train_samples": metadata["n_samples"]["train"],
                "test_samples": metadata["n_samples"]["test"],
                **metrics,
            })
        
        return results
    
    except Exception as e:
        print(f"Error processing {dataset_name} (mode={mode}, seed={seed}): {e}")
        return []


def evaluate_dataset(dataset_name, task_type, seeds, modes, n_jobs=-1):
    """Evaluate a dataset across all seeds and modes using parallel processing."""
    print(f"\nEvaluating {dataset_name} ({task_type})...")
    
    # Create all configurations to test
    configurations = [
        (dataset_name, mode, seed, task_type)
        for mode in modes
        for seed in seeds
    ]
    
    # Run evaluations in parallel
    all_results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(evaluate_single_configuration)(*config)
        for config in configurations
    )
    
    # Flatten results
    results = []
    for result_list in all_results:
        results.extend(result_list)
    
    return results

## Run Evaluation on All Datasets

In [9]:
# Collect all results
all_results = []

for _, row in available_datasets.iterrows():
    dataset_name = row["dataset_name"]
    task_type = row["task_type"]
    
    results = evaluate_dataset(
        dataset_name=dataset_name,
        task_type=task_type,
        seeds=SEEDS,
        modes=MODES,
        n_jobs=N_JOBS,
    )
    
    all_results.extend(results)

# Create results dataframe
results_df = pd.DataFrame(all_results)
print(f"\n{'='*60}")
print(f"Evaluation complete! Processed {len(results_df)} model runs")
print(f"Total datasets: {len(available_datasets)}")
print(f"Seeds per dataset: {len(SEEDS)}")
print(f"Modes per dataset: {len(MODES)}")
print(f"{'='*60}")


Evaluating iris (classification)...
Detected task type: classification
Detected 4 numerical features: ['num_sepal length', 'num_sepal width', 'num_petal length', 'num_petal width']
Detected 0 categorical features: []
Detected task type: classification
Detected 4 numerical features: ['num_sepal length', 'num_sepal width', 'num_petal length', 'num_petal width']
Detected 0 categorical features: []
Detected task type: classification
Detected task type: classification
Detected 4 numerical features: ['num_sepal length', 'num_sepal width', 'num_petal length', 'num_petal width']
Detected 0 categorical features: []
Detected 4 numerical features: ['num_sepal length', 'num_sepal width', 'num_petal length', 'num_petal width']
Detected 0 categorical features: []
Detected task type: classification
Detected 4 numerical features: ['num_sepal length', 'num_sepal width', 'num_petal length', 'num_petal width']
Detected 0 categorical features: []
Detected task type: classification
Detected 4 numerical fe

/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].clip(lower, upper).infer_objects(copy=False)
/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[

Detected task type: classification
Detected 6 numerical features: ['num_age', 'num_trestbps', 'num_chol', 'num_thalach', 'num_oldpeak']...
Detected 7 categorical features: ['cat_sex', 'cat_cp', 'cat_fbs', 'cat_restecg', 'cat_exang']...
Detected task type: classification
Detected 6 numerical features: ['num_age', 'num_trestbps', 'num_chol', 'num_thalach', 'num_oldpeak']...
Detected 7 categorical features: ['cat_sex', 'cat_cp', 'cat_fbs', 'cat_restecg', 'cat_exang']...
Detected task type: classification
Detected 6 numerical features: ['num_age', 'num_trestbps', 'num_chol', 'num_thalach', 'num_oldpeak']...
Detected 7 categorical features: ['cat_sex', 'cat_cp', 'cat_fbs', 'cat_restecg', 'cat_exang']...
Detected task type: classification
Detected 6 numerical features: ['num_age', 'num_trestbps', 'num_chol', 'num_thalach', 'num_oldpeak']...
Detected 7 categorical features: ['cat_sex', 'cat_cp', 'cat_fbs', 'cat_restecg', 'cat_exang']...
Detected task type: classification
Detected 6 numerical 

/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].clip(lower, upper).infer_objects(copy=False)
/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[

Detected task type: classification
Detected 23 numerical features: ['num_X1', 'num_X2', 'num_X3', 'num_X4', 'num_X5']...
Detected 0 categorical features: []


...


Evaluating real_estate_valuation (regression)...
Detected task type: regression
Detected 6 numerical features: ['num_X1 transaction date', 'num_X2 house age', 'num_X3 distance to the nearest MRT station', 'num_X4 number of convenience stores', 'num_X5 latitude']...
Detected 0 categorical features: []
Detected task type: regression
Detected 6 numerical features: ['num_X1 transaction date', 'num_X2 house age', 'num_X3 distance to the nearest MRT station', 'num_X4 number of convenience stores', 'num_X5 latitude']...
Detected 0 categorical features: []
Detected task type: regression
Detected 6 numerical features: ['num_X1 transaction date', 'num_X2 house age', 'num_X3 distance to the nearest MRT station', 'num_X4 number of convenience stores', 'num_X5 latitude']...
Detected 0 categorical features: []
Detected task type: regression
Detected 6 numerical features: ['num_X1 transaction date', 'num_X2 house age', 'num_X3 distance to the nearest MRT station', 'num_X4 number of convenience store

/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].clip(lower, upper).infer_objects(copy=False)
/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[

Detected task type: classification
Detected 6 numerical features: ['num_Fresh', 'num_Milk', 'num_Grocery', 'num_Frozen', 'num_Detergents_Paper']...
Detected 1 categorical features: ['cat_Channel']

Evaluating parkinsons (regression)...
Detected task type: classification
Detected 22 numerical features: ['num_MDVP:Fo', 'num_MDVP:Fhi', 'num_MDVP:Flo', 'num_MDVP:Jitter', 'num_MDVP:Jitter.1']...
Detected 0 categorical features: []
Detected task type: classification
Detected 22 numerical features: ['num_MDVP:Fo', 'num_MDVP:Fhi', 'num_MDVP:Flo', 'num_MDVP:Jitter', 'num_MDVP:Jitter.1']...
Detected 0 categorical features: []
Detected task type: classification
Detected 22 numerical features: ['num_MDVP:Fo', 'num_MDVP:Fhi', 'num_MDVP:Flo', 'num_MDVP:Jitter', 'num_MDVP:Jitter.1']...
Detected 0 categorical features: []
Detected task type: classification
Detected 22 numerical features: ['num_MDVP:Fo', 'num_MDVP:Fhi', 'num_MDVP:Flo', 'num_MDVP:Jitter', 'num_MDVP:Jitter.1']...
Detected 0 categorical f

/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].clip(lower, upper).infer_objects(copy=False)
/home/aarchetto/frog-dq/notebooks/../frogdq/preprocessing.py:323: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[

Detected task type: regression
Detected 10 numerical features: ['num_team', 'num_targeted_productivity', 'num_smv', 'num_wip', 'num_over_time']...
Detected 3 categorical features: ['cat_quarter', 'cat_department', 'cat_day']
Detected 1 date features: ['dat_date']
Detected task type: regression
Detected 10 numerical features: ['num_team', 'num_targeted_productivity', 'num_smv', 'num_wip', 'num_over_time']...
Detected 3 categorical features: ['cat_quarter', 'cat_department', 'cat_day']
Detected 1 date features: ['dat_date']
Detected task type: regression
Detected 10 numerical features: ['num_team', 'num_targeted_productivity', 'num_smv', 'num_wip', 'num_over_time']...
Detected 3 categorical features: ['cat_quarter', 'cat_department', 'cat_day']
Detected 1 date features: ['dat_date']
Detected task type: regression
Detected 10 numerical features: ['num_team', 'num_targeted_productivity', 'num_smv', 'num_wip', 'num_over_time']...
Detected 3 categorical features: ['cat_quarter', 'cat_departm

## Results Overview

In [10]:
# Display first few results
results_df.head(20)

,dataset,task_type,data_mode,seed,model,overall_quality,train_samples,test_samples,accuracy,f1_score,mse,r2_score
0,iris,classification,clean,42,LogisticRegression,100.0,90,30,0.933333,0.933333,NaN,NaN
1,iris,classification,clean,42,DecisionTree,100.0,90,30,0.933333,0.933333,NaN,NaN
2,iris,classification,clean,42,RandomForest,100.0,90,30,0.933333,0.933333,NaN,NaN
3,iris,classification,clean,42,KNN,100.0,90,30,0.966667,0.966583,NaN,NaN
4,iris,classification,clean,43,LogisticRegression,100.0,90,30,0.966667,0.966583,NaN,NaN
5,iris,classification,clean,43,DecisionTree,100.0,90,30,1.000000,1.000000,NaN,NaN
6,iris,classification,clean,43,RandomForest,100.0,90,30,0.966667,0.966583,NaN,NaN
7,iris,classification,clean,43,KNN,100.0,90,30,0.933333,0.932660,NaN,NaN
8,iris,classification,clean,44,LogisticRegression,100.0,90,30,0.966667,0.966583,NaN,NaN
9,iris,classification,clean,44,DecisionTree,100.0,90,30,0.966667,0.966583,NaN,NaN


## Summary Statistics Across Seeds

In [12]:
# Summary statistics for classification tasks
if "accuracy" in results_df.columns:
    cls_results = results_df[results_df["task_type"] == "classification"].copy()

    print("\n" + "=" * 60)
    print("CLASSIFICATION TASKS SUMMARY (Averaged Across Seeds)")
    print("=" * 60)

    summary = cls_results.groupby(["data_mode", "model"])[["accuracy", "f1_score"]].agg(
        ["mean", "std"]
    )
    print(summary)


CLASSIFICATION TASKS SUMMARY (Averaged Across Seeds)
                              accuracy            f1_score          
                                  mean       std      mean       std
data_mode model                                                     
ar        DecisionTree        0.769501  0.196240  0.753498  0.227555
          KNN                 0.788892  0.195531  0.768150  0.223002
          LogisticRegression  0.799948  0.179302  0.773704  0.216741
          RandomForest        0.818163  0.177725  0.790882  0.218754
clean     DecisionTree        0.812184  0.167322  0.805942  0.176831
          KNN                 0.814504  0.171011  0.801425  0.183684
          LogisticRegression  0.829758  0.164176  0.813245  0.182373
          RandomForest        0.853044  0.153199  0.837334  0.176084
nar       DecisionTree        0.757028  0.200328  0.742794  0.224326
          KNN                 0.781969  0.192176  0.761773  0.220063
          LogisticRegression  0.796087  0.182043 

In [13]:
# Summary statistics for regression tasks
if "r2_score" in results_df.columns:
    reg_results = results_df[results_df["task_type"] == "regression"].copy()

    print("\n" + "=" * 60)
    print("REGRESSION TASKS SUMMARY (Averaged Across Seeds)")
    print("=" * 60)

    summary = reg_results.groupby(["data_mode", "model"])[["r2_score"]].agg(
        ["mean", "std"]
    )
    print(summary)


REGRESSION TASKS SUMMARY (Averaged Across Seeds)
                        r2_score          
                            mean       std
data_mode model                           
ar        DecisionTree  0.301235  0.491534
          KNN           0.484427  0.316563
          RandomForest  0.540666  0.308633
          Ridge         0.436411  0.302851
clean     DecisionTree  0.126492  1.990773
          KNN           0.479150  0.515825
          RandomForest  0.526416  0.631822
          Ridge         0.490285  0.300226
nar       DecisionTree  0.063664  1.758892
          KNN           0.420634  0.535404
          RandomForest  0.458644  0.526925
          Ridge         0.424649  0.301014


## Performance Degradation Analysis

In [14]:
# Calculate performance degradation for classification
if "accuracy" in results_df.columns:
    cls_results = results_df[results_df["task_type"] == "classification"].copy()

    print("\n" + "=" * 60)
    print("CLASSIFICATION: Performance Degradation (Accuracy)")
    print("=" * 60)

    # Create pivot table with multi-level aggregation
    pivot = cls_results.pivot_table(
        index=["dataset", "model", "seed"],
        columns="data_mode",
        values="accuracy",
    )

    if "clean" in pivot.columns:
        if "ar" in pivot.columns:
            pivot["ar_degradation"] = pivot["clean"] - pivot["ar"]
        if "nar" in pivot.columns:
            pivot["nar_degradation"] = pivot["clean"] - pivot["nar"]

    print("\nDegradation Statistics (averaged across seeds and datasets):")
    print(pivot[[col for col in pivot.columns if "degradation" in col]].describe())
    
    # Average across seeds for cleaner view
    pivot_avg = pivot.groupby(["dataset", "model"]).mean()
    print("\nTop 10 datasets with highest NAR degradation:")
    if "nar_degradation" in pivot_avg.columns:
        top_degraded = pivot_avg.nlargest(10, "nar_degradation")
        print(top_degraded[["clean", "nar", "nar_degradation"]])


CLASSIFICATION: Performance Degradation (Accuracy)

Degradation Statistics (averaged across seeds and datasets):
data_mode  ar_degradation  nar_degradation
count          680.000000       680.000000
mean             0.033246         0.041368
std              0.111327         0.116823
min             -0.153846        -0.153846
25%              0.000000         0.000000
50%              0.003686         0.008980
75%              0.028120         0.046922
max              0.809524         0.857143

Top 10 datasets with highest NAR degradation:
data_mode                                                                 clean  \
dataset                                            model                          
zoo                                                DecisionTree        0.933333   
                                                   RandomForest        0.961905   
                                                   KNN                 0.857143   
                                     

In [15]:
# Calculate performance degradation for regression
if "r2_score" in results_df.columns:
    reg_results = results_df[results_df["task_type"] == "regression"].copy()

    print("\n" + "=" * 60)
    print("REGRESSION: Performance Degradation (R2 Score)")
    print("=" * 60)

    pivot = reg_results.pivot_table(
        index=["dataset", "model", "seed"],
        columns="data_mode",
        values="r2_score",
    )

    if "clean" in pivot.columns:
        if "ar" in pivot.columns:
            pivot["ar_degradation"] = pivot["clean"] - pivot["ar"]
        if "nar" in pivot.columns:
            pivot["nar_degradation"] = pivot["clean"] - pivot["nar"]

    print("\nDegradation Statistics (averaged across seeds and datasets):")
    print(pivot[[col for col in pivot.columns if "degradation" in col]].describe())
    
    # Average across seeds for cleaner view
    pivot_avg = pivot.groupby(["dataset", "model"]).mean()
    print("\nTop 10 datasets with highest NAR degradation:")
    if "nar_degradation" in pivot_avg.columns:
        top_degraded = pivot_avg.nlargest(10, "nar_degradation")
        print(top_degraded[["clean", "nar", "nar_degradation"]])


REGRESSION: Performance Degradation (R2 Score)

Degradation Statistics (averaged across seeds and datasets):
data_mode  ar_degradation  nar_degradation
count          320.000000       320.000000
mean            -0.035099         0.063688
std              0.957131         0.238110
min            -16.018831        -2.044379
25%              0.000162         0.010642
50%              0.016722         0.037646
75%              0.049855         0.093786
max              0.592850         1.604443

Top 10 datasets with highest NAR degradation:
data_mode                                     clean       nar  nar_degradation
dataset                      model                                            
bike_sharing                 RandomForest  0.743012  0.279847         0.463165
                             DecisionTree  0.658940  0.207832         0.451108
                             Ridge         0.680379  0.241150         0.439229
                             KNN           0.555143  0.235001

## Model Comparison Across Data Modes

In [16]:
# Average performance by model and data mode (across all seeds)
if "accuracy" in results_df.columns:
    print("\nClassification: Average Accuracy by Model and Data Mode (Across Seeds)")
    print("=" * 60)
    cls_pivot = results_df[results_df["task_type"] == "classification"].pivot_table(
        index="model", columns="data_mode", values="accuracy", aggfunc="mean"
    )
    print(cls_pivot.round(4))

if "r2_score" in results_df.columns:
    print("\nRegression: Average R2 Score by Model and Data Mode (Across Seeds)")
    print("=" * 60)
    reg_pivot = results_df[results_df["task_type"] == "regression"].pivot_table(
        index="model", columns="data_mode", values="r2_score", aggfunc="mean"
    )
    print(reg_pivot.round(4))


Classification: Average Accuracy by Model and Data Mode (Across Seeds)
data_mode               ar   clean     nar
model                                     
DecisionTree        0.7695  0.8122  0.7570
KNN                 0.7889  0.8145  0.7820
LogisticRegression  0.7999  0.8298  0.7961
RandomForest        0.8182  0.8530  0.8089

Regression: Average R2 Score by Model and Data Mode (Across Seeds)
data_mode         ar   clean     nar
model                               
DecisionTree  0.3012  0.1265  0.0637
KNN           0.4844  0.4792  0.4206
RandomForest  0.5407  0.5264  0.4586
Ridge         0.4364  0.4903  0.4246


## Variance Across Seeds

In [17]:
# Analyze variance in results across different seeds
print("\n" + "=" * 60)
print("VARIANCE ACROSS SEEDS")
print("=" * 60)

if "accuracy" in results_df.columns:
    cls_variance = results_df[results_df["task_type"] == "classification"].groupby(
        ["dataset", "data_mode", "model"]
    )["accuracy"].agg(["mean", "std"])
    
    print("\nClassification - Datasets with highest variance across seeds:")
    print(cls_variance.nlargest(10, "std"))

if "r2_score" in results_df.columns:
    reg_variance = results_df[results_df["task_type"] == "regression"].groupby(
        ["dataset", "data_mode", "model"]
    )["r2_score"].agg(["mean", "std"])
    
    print("\nRegression - Datasets with highest variance across seeds:")
    print(reg_variance.nlargest(10, "std"))


VARIANCE ACROSS SEEDS

Classification - Datasets with highest variance across seeds:
                                                                                   mean  \
dataset                                          data_mode model                          
national_poll_on_healthy_aging_npha              ar        DecisionTree        0.409790   
                                                           KNN                 0.455944   
                                                 nar       KNN                 0.453147   
zoo                                              ar        KNN                 0.295238   
                                                 nar       KNN                 0.295238   
national_poll_on_healthy_aging_npha              nar       RandomForest        0.476923   
                                                           LogisticRegression  0.476923   
higher_education_students_performance_evaluation nar       DecisionTree        0.200000   
glas